In [1]:
import torch
import torch.nn.functional as F

# ============== 输入设置 ==============
# 两个示例序列:
# 序列1: [1, 2, 0] (0是填充符号)
# 序列2: [3, 4, 5] (没有填充)
inputs = torch.tensor([[1, 2, 0], [3, 4, 5]])
print("输入序列:")
print(inputs)

# ============== 1. 填充掩码 ==============
print("\n== 第一部分: 填充掩码 ==")

# 填充掩码标识哪些位置包含填充符(0)
# True = 这个位置应被忽略
# False = 这个位置包含真实数据
key_padding_mask = (inputs == 0)
print("\n填充掩码 (True表示'忽略此位置'):")
print(key_padding_mask)
print("解释: 第一个序列的最后一位是填充符，所以标记为True")

# ============== 2. 因果掩码 ==============
print("\n== 第二部分: 因果掩码 ==")

# 因果掩码防止token关注未来位置
# 这对GPT等自回归模型至关重要
seq_len = inputs.size(1)

# 创建下三角矩阵(包括对角线)，1表示可见，0表示被掩盖
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
# 转换为布尔掩码(0->True表示需掩盖，1->False表示可见)
causal_mask = causal_mask == 0
print("\n因果掩码 (True表示'需掩盖'):")
print(causal_mask)
print("解释: 每个位置只能看到自己和之前的位置")

# ============== 3. 应用掩码到注意力分数 ==============
print("\n== 第三部分: 应用掩码 ==")

# 模拟注意力分数
attn_scores = torch.randn(2, 3, 3)
print("\n原始注意力分数 (随机):")
print(attn_scores)

# 应用填充掩码
attn_scores = attn_scores.masked_fill(
    key_padding_mask.unsqueeze(1),  # [2,3] -> [2,1,3]
    -1e9  # 非常小的值，softmax后接近0
)

# 应用因果掩码
attn_scores = attn_scores.masked_fill(
    causal_mask.unsqueeze(0),  # [3,3] -> [1,3,3]
    -1e9
)

# 计算最终注意力权重
attn_weights = F.softmax(attn_scores, dim=-1)
print("\n最终注意力权重:")
print(attn_weights)
print("解释: 权重已被两种掩码调整，填充位置和未来位置的权重接近0")


输入序列:
tensor([[1, 2, 0],
        [3, 4, 5]])

== 第一部分: 填充掩码 ==

填充掩码 (True表示'忽略此位置'):
tensor([[False, False,  True],
        [False, False, False]])
解释: 第一个序列的最后一位是填充符，所以标记为True

== 第二部分: 因果掩码 ==

因果掩码 (True表示'需掩盖'):
tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
解释: 每个位置只能看到自己和之前的位置

== 第三部分: 应用掩码 ==

原始注意力分数 (随机):
tensor([[[ 0.3708,  0.5389,  0.9347],
         [-0.8493, -0.1082,  1.8963],
         [-0.3475,  0.0124, -2.1640]],

        [[ 0.4060,  0.2815, -1.3372],
         [ 2.0263, -0.2249,  0.4969],
         [-0.4996, -1.1046, -1.1554]]])

最终注意力权重:
tensor([[[1.0000, 0.0000, 0.0000],
         [0.3228, 0.6772, 0.0000],
         [0.4110, 0.5890, 0.0000]],

        [[1.0000, 0.0000, 0.0000],
         [0.9048, 0.0952, 0.0000],
         [0.4842, 0.2644, 0.2513]]])
解释: 权重已被两种掩码调整，填充位置和未来位置的权重接近0


In [7]:
import torch
import torch.nn as nn

# 设置随机种子，确保实验结果可复现
torch.random.manual_seed(42)

# 定义模型维度和序列长度
d_model = 7  # 模型的维度
tgt_len = 6  # 目标序列长度
src_len = 5  # 源序列长度

# =================== 编码器部分 ===================
# 随机生成源序列的嵌入表示
src_em_input = torch.randn(src_len, d_model)
# 定义线性变换层用于计算Q、K、V
q_proj0 = nn.Linear(d_model, d_model)
k_proj0 = nn.Linear(d_model, d_model)
v_proj0 = nn.Linear(d_model, d_model)
# 计算Q、K、V
q, k, v = q_proj0(src_em_input), k_proj0(src_em_input), v_proj0(src_em_input)
# 创建源序列的填充掩码
src_key_padding_mask = torch.tensor([[False, False, False, True, True]])
# 计算注意力权重
weight = q @ k.transpose(0, 1)  # [src_len, src_len]
print("Encoder attention weight before key padding:")
print(weight)
# 应用填充掩码
weight = weight.masked_fill(src_key_padding_mask, float('-inf'))
print("Encoder attention weight after key padding:")
print(weight)
# 计算softmax值
weight = torch.softmax(weight, dim=-1)  # [src_len, src_len]
# 得到加权后的输出（记忆）
memory = weight @ v  # [src_len, d_model]

# =================== 解码器部分 ===================
## ------------------ 带遮罩的自注意力机制
# 随机生成目标序列的嵌入表示
tgt_em_input = torch.randn(tgt_len, d_model)
# 创建目标序列的填充掩码
tgt_key_padding_mask = torch.tensor([[False, False, False, True, True, True]])
# 创建上三角矩阵作为遮罩，防止未来信息泄露
attn_mask = (torch.triu(torch.ones((tgt_len, tgt_len))) == 1).transpose(0, 1)
attn_mask = attn_mask.float().masked_fill(attn_mask == 0, float('-inf')).masked_fill(attn_mask == 1, float(0.))
# 定义线性变换层用于计算Q、K、V
q_proj1 = nn.Linear(d_model, d_model)
k_proj1 = nn.Linear(d_model, d_model)
v_proj1 = nn.Linear(d_model, d_model)
# 计算Q、K、V
q, k, v = q_proj1(tgt_em_input), k_proj1(tgt_em_input), v_proj1(tgt_em_input)  # [tgt_len, d_model]
# 计算注意力权重
weight = q @ k.transpose(0, 1)  # [tgt_len, tgt_len]
weight += attn_mask
print("Decoder masked attention weight before key padding:")
print(weight)



# 这里控制是否应用填充掩码
weight = weight.masked_fill(tgt_key_padding_mask, float('-inf'))
print("Decoder masked attention weight after key padding:")
print(weight)

weight = torch.softmax(weight, dim=-1)  # [tgt_len, tgt_len]
output = weight @ v  # [tgt_len, tgt_len] @ [tgt_len, d_model]



## ------------------ 交叉注意力机制
# 定义线性变换层用于计算Q、K、V
q_proj2 = nn.Linear(d_model, d_model)
k_proj2 = nn.Linear(d_model, d_model)
v_proj2 = nn.Linear(d_model, d_model)
# 计算Q、K、V，其中K和V来自编码器的记忆
q, k, v = q_proj2(output), k_proj2(memory), v_proj2(memory)  # q: [tgt_len, d_model], k,v: [src_len, d_model]
# 计算注意力权重
weight = q @ k.transpose(0, 1)  # [tgt_len, src_len]
# 应用源序列的填充掩码
weight = weight.masked_fill(src_key_padding_mask, float('-inf'))
weight = torch.softmax(weight, dim=-1)
output = weight @ v  # [tgt_len, d_model]

# =================== 损失计算 ===================
# 定义目标标签和损失函数
tgt = torch.tensor([1, 2, 3, 0, 0, 0], dtype=torch.long)
loss_fcn = torch.nn.CrossEntropyLoss()  # 忽略填充标记（这里为0）的损失计算
loss = loss_fcn(output, tgt)
print(loss)


Encoder attention weight before key padding:
tensor([[ 4.1716e+00,  8.0356e-01, -3.5906e-01, -2.8543e+00, -1.0852e+00],
        [ 1.0268e+00, -1.4225e-01, -7.9872e-01, -1.0038e+00, -4.1425e-01],
        [ 1.8088e+00, -3.1054e-02, -1.9290e+00, -1.2489e+00, -5.8257e-01],
        [ 4.8749e-01,  3.1420e-01, -6.9572e-01,  7.4465e-04,  6.1609e-01],
        [ 3.7340e+00, -2.8372e-02, -1.4023e+00, -1.6844e+00,  6.6219e-01]],
       grad_fn=<MmBackward0>)
Encoder attention weight after key padding:
tensor([[ 4.1716,  0.8036, -0.3591,    -inf,    -inf],
        [ 1.0268, -0.1422, -0.7987,    -inf,    -inf],
        [ 1.8088, -0.0311, -1.9290,    -inf,    -inf],
        [ 0.4875,  0.3142, -0.6957,    -inf,    -inf],
        [ 3.7340, -0.0284, -1.4023,    -inf,    -inf]],
       grad_fn=<MaskedFillBackward0>)
Decoder masked attention weight before key padding:
tensor([[-1.2846,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0381, -0.1050,    -inf,    -inf,    -inf,    -inf],
        [ 0